# [SITCOM-1593] - M1M3 Force actuator analysis for different elevations

Following [SITCOM-1593], we want to plot the forces actuator errors as a function of elevation. 

Given a time range, this notebook will plot the average primaryCylinderFollowingError as well as the maximum and minimum, for different elevations. 

[SITCOM-1593]: https://rubinobs.atlassian.net/browse/SITCOM-1593

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.time import Time
from pathlib import Path

from lsst.summit.utils.efdUtils import EfdClient, getEfdData, makeEfdClient
from lsst.sitcom.vandv import m1m3
from lsst.ts.xml.tables.m1m3 import FATable
from lsst.ts.xml.enums.MTM1M3 import DetailedStates

from collections import defaultdict
from bokeh.plotting import figure, show
from bokeh.models import HoverTool, TapTool
from bokeh.layouts import column
from bokeh.models import ColumnDataSource
from bokeh.io import output_notebook

N_ACTUATORS = 156


In [ ]:
# resampling value for the loaded dataframe in seconds
resample_in_sec = 60  
# choose the number of bins in which we want to group the elevation_resampled values (x axis)
n_bins = 30 

In [ ]:
client = makeEfdClient()

In [ ]:
print(FATable[0].actuator_id)

In [ ]:
start = Time("2024-11-09 02:15:0Z", scale="utc")
end = Time("2024-11-09 05:15:0Z", scale="utc")
topic = f"lsst.sal.MTM1M3.forceActuatorData"

In [ ]:
plot_name = "Force Actuator Error vs Elevation"
plot_path = Path("./plots")
plot_path.mkdir(exist_ok=True, parents=True)

### retrieve data from EFD

In [ ]:
# Tested with up to 3h of data in USDF, used 16GB node
# Retrieve Force Actuator information
primary_FA_error = [f"primaryCylinderFollowingError{i}" for i in range(N_ACTUATORS)]
df_primary_FA_error = getEfdData(
    client,"lsst.sal.MTM1M3.forceActuatorData", columns=primary_FA_error, begin=start, end=end
)  
# Retrieve elevations
elevations = getEfdData(
    client,"lsst.sal.MTMount.elevation", columns="actualPosition", begin=start, end=end
)


### resample the data into more manageable time chunks

In [ ]:
# take the mean value for each of the actuators in {resample_in_sec} second samples
df_primary_FA_error_resampled_mean = df_primary_FA_error.resample(
    f"{resample_in_sec}s"
).mean()

# take the maximum value for each of the actuators in those {resample_in_sec} second samples
df_primary_FA_error_resampled_max = df_primary_FA_error.resample(
    f"{resample_in_sec}s"
).max()

# take the minimum value for each of the actuators in those {resample_in_sec} second samples
df_primary_FA_error_resampled_min = df_primary_FA_error.resample(
    f"{resample_in_sec}s"
).min()

# mean value of elevation in the time period ({resample_in_sec} seconds)
elevations_resampled = (
    elevations["actualPosition"].resample(f"{resample_in_sec}s").mean()
)

### obtain average, maximum and minimum across the 156 actuators

In [ ]:
average_across_actuators_resampled = df_primary_FA_error_resampled_mean.mean(axis=1)
max_across_actuators_resampled = df_primary_FA_error_resampled_max.max(axis=1)
min_across_actuators_resampled = df_primary_FA_error_resampled_min.min(axis=1)

### make a scatter plot of force actuator errors for different elevations
Note that some elevation values might be repeated or slightly offset from other, similar values as the sequence of elevation stops can change a lot

In [ ]:
bin_edges = np.linspace(
    np.min(elevations_resampled), np.max(elevations_resampled), n_bins + 1
)
bin_indices = np.digitize(elevations_resampled, bin_edges)

bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

plt.scatter(
    elevations_resampled,
    average_across_actuators_resampled,
    alpha=0.3,
    color="blue",
    label=f"Average value per {resample_in_sec}s period",
    s=5,
)

plt.scatter(
    elevations_resampled,
    max_across_actuators_resampled,
    alpha=0.3,
    color="orange",
    label=f"Largest value per {resample_in_sec}s period",
    s=5,
)

plt.scatter(
    elevations_resampled,
    min_across_actuators_resampled,
    alpha=0.3,
    color="red",
    label=f"Lowest value per {resample_in_sec}s period",
    s=5,
)

plt.title(plot_name, y=1.08)
t0 = pd.to_datetime(start.datetime, utc=True)
t1 = pd.to_datetime(end.datetime, utc=True)
plt.suptitle(f"{t0} - {t1}", y=0.93, fontsize=10, color="gray")
plt.ylabel("primaryCylinderFollowingError (N)")
plt.xlabel("elevation (degrees)")
plt.legend()
plt.savefig(plot_path / "sitcom-1593_fa_error_vs_elevation_scatter.png")

### histogram the actuator ID that hits the maximum and minimum record most often

In [ ]:
# get the actuator ID which has a maximum or minimum value in the resampled data set
# with maximum and minimum values of each actuator

#first determine an array storing the actuator_id values
aid = np.empty(len(FATable))
for i in range(len(FATable)):
    aid[i] = FATable[i].actuator_id

max_actuators = np.argmax(df_primary_FA_error_resampled_max, axis=1)
max_actuators_id = np.empty(len(max_actuators))
min_actuators = np.argmin(df_primary_FA_error_resampled_min, axis=1)
min_actuators_id = np.empty(len(min_actuators))
for i in range(len(max_actuators)):
    max_actuators_id[i] = FATable[max_actuators[i]].actuator_id
    min_actuators_id[i] = FATable[min_actuators[i]].actuator_id

minhist = plt.hist(
    min_actuators_id, bins=156, range=[np.min(aid), np.max(aid)], color="red", label="Minimum"
)
maxhist = plt.hist(
    max_actuators_id, bins=156, range=[np.min(aid), np.max(aid)], color="orange", label="Maximum"
) 

plt.xlabel("Actuator ID")
plt.yscale('log')
plt.grid(axis="y",which="minor")
#add grid lines
plt.legend()
plt.savefig(plot_path / "sitcom-1593_actuator_id_hist.png")

In [ ]:
aid = np.empty(len(FATable))
for i in range(len(FATable)):
    aid[i] = FATable[i].actuator_id

print(np.max(aid),np.min(aid))

### plot all actuator force error behavior as a function of elevation

In [ ]:
binned_primary_FA_error_resampled_mean = [None] * N_ACTUATORS
binned_primary_FA_error_resampled_max = [None] * N_ACTUATORS
binned_primary_FA_error_resampled_min = [None] * N_ACTUATORS
bin_edges = np.linspace(
    np.min(elevations_resampled), np.max(elevations_resampled), n_bins + 1
)
bin_indices = np.digitize(elevations_resampled, bin_edges)

for j in range(N_ACTUATORS):
    binned_primary_FA_error_resampled_mean[j] = [
        df_primary_FA_error_resampled_mean[bin_indices == i][
            f"primaryCylinderFollowingError{j}"
        ].mean()
        for i in range(1, len(bin_edges))
    ]
    binned_primary_FA_error_resampled_max[j] = [
        df_primary_FA_error_resampled_max[bin_indices == i][
            f"primaryCylinderFollowingError{j}"
        ].mean()
        for i in range(1, len(bin_edges))
    ]
    binned_primary_FA_error_resampled_min[j] = [
        df_primary_FA_error_resampled_min[bin_indices == i][
            f"primaryCylinderFollowingError{j}"
        ].mean()
        for i in range(1, len(bin_edges))
    ]
    
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2


In [ ]:
for j in range(N_ACTUATORS):
    plt.plot(
        bin_centers,
        binned_primary_FA_error_resampled_mean[j],
        alpha=0.1,
        color="gray",
    )

    plt.plot(
        bin_centers,
        binned_primary_FA_error_resampled_min[j],
        alpha=0.1,
        color="red",
    )

    plt.plot(
        bin_centers,
        binned_primary_FA_error_resampled_max[j],
        alpha=0.1,
        color="orange",
    )

plt.title(f"{plot_name}: all actuators", y=1.08)
t0 = pd.to_datetime(start.datetime, utc=True)
t1 = pd.to_datetime(end.datetime, utc=True)
plt.suptitle(f"{t0} - {t1}", y=0.93, fontsize=10, color="gray")
plt.ylabel("primaryCylinderFollowingError (N)")
plt.xlabel("elevation (degrees)")
plt.legend()
plt.savefig(plot_path / "sitcom-1593_fa_error_vs_elevation_line.png")

### add interactivity to the plot
So that we can identify with hover the actuator IDs

In [ ]:
fadata_mean = defaultdict(list)
fadata_max = defaultdict(list)
fadata_min = defaultdict(list)

fa_mean = np.array(binned_primary_FA_error_resampled_mean)
fa_max = np.array(binned_primary_FA_error_resampled_max)
fa_min = np.array(binned_primary_FA_error_resampled_min)

for i in range(N_ACTUATORS):
    actuator_id = FATable[i].actuator_id
    fadata_mean["elevation"].append(bin_centers)
    fadata_mean["force error"].append(fa_mean[i])
    fadata_mean["FA"].append(f"actuator_id {actuator_id}")
    fadata_max["elevation"].append(bin_centers)
    fadata_max["force error"].append(fa_max[i])
    fadata_max["FA"].append(f"actuator_id {actuator_id}")
    fadata_min["elevation"].append(bin_centers)
    fadata_min["force error"].append(fa_min[i])
    fadata_min["FA"].append(f"actuator_id {actuator_id}")

hover_opts = dict(tooltips=[("FA", "@FA")], show_arrow=False, line_policy="next")
line_opts_mean = dict(
    line_width=1,
    line_color="grey",
    line_alpha=0.1,
    hover_line_alpha=1.0,
    source=fadata_mean,
)
line_opts_max = dict(
    line_width=1,
    line_color="orange",
    line_alpha=0.1,
    hover_line_alpha=1.0,
    source=fadata_max,
)
line_opts_min = dict(
    line_width=1,
    line_color="red",
    line_alpha=0.1,
    hover_line_alpha=1.0,
    source=fadata_min,
)

p = figure(
    title=f"{plot_name}: all actuators",
    x_axis_label="elevation (degrees)",
    y_axis_label="primaryCylinderFollowingError (N)",
    tools=[HoverTool(**hover_opts), TapTool()],
)

p.multi_line(xs="elevation", ys="force error", **line_opts_mean)
p.multi_line(xs="elevation", ys="force error", **line_opts_max)
p.multi_line(xs="elevation", ys="force error", **line_opts_min)

output_notebook()
show(p)